# Adult Income Prediction

**Tabular Classification Project**

## 2 · Project Overview

Predict whether an individual earns **>$50K/year** based on census attributes (age, education, occupation, hours-per-week, etc.). A classic binary classification benchmark with ~32K rows and mixed feature types.

## 3 · Learning Objectives

1. Perform EDA and target analysis on a classification dataset.
2. Handle categorical encoding, missing values, and class imbalance.
3. Build a baseline model and compare against modern boosting models.
4. Use LazyPredict for rapid classifier benchmarking.
5. Run FLAML AutoML for automated model selection.
6. Train CatBoost, LightGBM, and XGBoost with GPU auto-detection.
7. Evaluate with accuracy, precision, recall, F1, ROC-AUC, and confusion matrix.

## 4 · Problem Statement

Given demographic and employment attributes from census data, classify whether a person's annual income exceeds $50,000.

## 5 · Why This Project Matters

- Income prediction models are used in **credit scoring**, **ad targeting**, and **policy analysis**.
- This dataset exposes important ML concepts: class imbalance (~24% positive), mixed feature types, and feature importance.
- It teaches responsible ML — features like race and sex raise **fairness concerns**.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| Rows | ~32,561 |
| Features | 14 (6 numeric, 8 categorical) |
| Target | `income` (binary: <=50K / >50K) |
| Class balance | ~76% <=50K, ~24% >50K |
| Missing values | ~7% in workclass, occupation, native-country |

## 7 · Dataset Source and License Notes

**Source**: UCI ML Repository — Adult (Census Income) dataset.
**License**: Public domain / educational.
**Citation**: Becker & Kohavi, 1996.

## 8 · Environment Setup

In [1]:
import subprocess, sys

def _install(pkg):
    try:
        __import__(pkg.replace('-','_'))
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["catboost", "lightgbm", "xgboost", "flaml", "lazypredict"]:
    _install(pkg)
print("All packages ready.")

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## 9 · Imports

In [2]:
import os, json, time, warnings, random
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix,
                              classification_report, ConfusionMatrixDisplay)

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Imports complete.")

Imports complete.


## 10 · Configuration / Constants

In [3]:
TARGET = "income"
SEED = 42
SAVE_DIR = os.path.dirname(os.path.abspath(__file__)) if '__file__' in dir() else '.'
SAVE_DIR = os.path.dirname(os.path.abspath('__dummy__'))
print(f"Save dir: {SAVE_DIR}")

Save dir: E:\Github\Machine-Learning-Projects\Classification\Adult Income Prediction


## 11 · Dataset Download or Loading

In [4]:
from sklearn.datasets import fetch_openml

data = fetch_openml('adult', version=2, as_frame=True, parser='auto')
df = data.frame.copy()
# Target column is 'class' in OpenML
df = df.rename(columns={'class': 'income'})
df['income'] = df['income'].astype(str).str.strip().str.replace('.', '', regex=False)
df['income'] = (df['income'] == '>50K').astype(int)
print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (48842, 15)


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,0
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,0
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,1
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,1
4,18,NaN,103497,Some-college,10,Never-married,NaN,Own-child,White,Female,0,0,30,United-States,0


## 12 · Data Validation Checks

In [5]:
print("=" * 50)
print("DATA VALIDATION")
print("=" * 50)
print(f"\nShape: {df.shape}")
print(f"\nMissing values:\n{df.isnull().sum()[df.isnull().sum() > 0]}")
if df.isnull().sum().sum() == 0:
    print("No missing values.")
print(f"\nDuplicate rows: {df.duplicated().sum()}")
print(f"\nTarget distribution:\n{df[TARGET].value_counts()}")
print(f"\nDtypes:\n{df.dtypes}")
assert TARGET in df.columns, f"Target '{TARGET}' not found!"
print(f"\nTarget '{TARGET}' confirmed.")

DATA VALIDATION

Shape: (48842, 15)

Missing values:
workclass         2799
occupation        2809
native-country     857
dtype: int64

Duplicate rows: 52

Target distribution:
income
0    37155
1    11687
Name: count, dtype: int64

Dtypes:
age                  int64
workclass         category
fnlwgt               int64
education         category
education-num        int64
marital-status    category
occupation        category
relationship      category
race              category
sex               category
capital-gain         int64
capital-loss         int64
hours-per-week       int64
native-country    category
income               int64
dtype: object

Target 'income' confirmed.


## 13 · Exploratory Data Analysis

In [6]:
num_cols = df.select_dtypes(include='number').columns.tolist()
cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for i, col in enumerate(num_cols[:6]):
    ax = axes[i // 3, i % 3]
    df[col].hist(bins=30, ax=ax, edgecolor='black', alpha=0.7)
    ax.set_title(col)
plt.suptitle("Numeric Feature Distributions", fontsize=14)
plt.tight_layout()
plt.savefig('eda_numeric.png', dpi=100, bbox_inches='tight')
plt.show()

# Correlation
corr = df[num_cols].corr()
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Numeric Feature Correlations")
plt.tight_layout()
plt.show()
print(f"Numeric features: {len(num_cols)}, Categorical features: {len(cat_cols)}")

Numeric features: 7, Categorical features: 8


## 14 · Target Analysis

In [7]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
df[TARGET].value_counts().plot(kind='bar', ax=axes[0], color=['steelblue', 'coral'], edgecolor='black')
axes[0].set_title("Target Distribution")
axes[0].set_xticklabels(['<=50K (0)', '>50K (1)'], rotation=0)

df[TARGET].value_counts(normalize=True).plot(kind='pie', ax=axes[1], autopct='%1.1f%%', colors=['steelblue', 'coral'])
axes[1].set_title("Class Proportions")
axes[1].set_ylabel('')
plt.tight_layout()
plt.show()

print(f"Class distribution:\n{df[TARGET].value_counts()}")
print(f"\nImbalance ratio: {df[TARGET].value_counts().iloc[0] / df[TARGET].value_counts().iloc[1]:.2f}:1")

Class distribution:
income
0    37155
1    11687
Name: count, dtype: int64

Imbalance ratio: 3.18:1


## 15 · Train/Validation/Test Split Strategy

Stratified 80/20 split to preserve class distribution.

In [8]:
# Handle missing values
df = df.dropna().reset_index(drop=True)

# Encode categoricals
cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
df[cat_cols] = oe.fit_transform(df[cat_cols])

X = df.drop(columns=[TARGET])
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train target dist: {y_train.value_counts().to_dict()}")

Train: (36177, 14), Test: (9045, 14)
Train target dist: {0: 27211, 1: 8966}


## 16 · Preprocessing Strategy

Categorical features encoded via OrdinalEncoder. Missing values handled before split. Tree-based models handle raw features without scaling.

## 17 · Feature Engineering

In [9]:
# Clean column names for LightGBM/XGBoost compatibility
clean = [c.replace('-', '_').replace(' ', '_').replace('.', '_') for c in X_train.columns]
X_train.columns = clean
X_test.columns = clean
print(f"Features ({len(clean)}): {clean[:10]}...")

Features (14): ['age', 'workclass', 'fnlwgt', 'education', 'education_num', 'marital_status', 'occupation', 'relationship', 'race', 'sex']...


## 18 · Baseline Model

Logistic Regression as baseline.

In [10]:
baseline = LogisticRegression(max_iter=1000, random_state=SEED)
baseline.fit(X_train, y_train)
y_pred_bl = baseline.predict(X_test)
y_prob_bl = baseline.predict_proba(X_test)[:, 1] if len(np.unique(y_train)) == 2 else None

print("=" * 50)
print("BASELINE: Logistic Regression")
print("=" * 50)
print(f"Accuracy : {accuracy_score(y_test, y_pred_bl):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_bl, average='weighted'):.4f}")
print(f"Recall   : {recall_score(y_test, y_pred_bl, average='weighted'):.4f}")
print(f"F1       : {f1_score(y_test, y_pred_bl, average='weighted'):.4f}")
if y_prob_bl is not None:
    print(f"ROC-AUC  : {roc_auc_score(y_test, y_prob_bl):.4f}")

BASELINE: Logistic Regression
Accuracy : 0.8015
Precision: 0.7858
Recall   : 0.8015
F1       : 0.7813
ROC-AUC  : 0.8076


## 19 · LazyPredict Benchmark

In [11]:
from lazypredict.Supervised import LazyClassifier

lazy = LazyClassifier(verbose=0, ignore_warnings=True)
lazy_models, _ = lazy.fit(X_train, X_test, y_train, y_test)
print("LazyPredict — Top 15 Classifiers:")
print(lazy_models.head(15).to_string())

LazyPredict — Top 15 Classifiers:
                        Accuracy  Balanced Accuracy   ROC AUC  F1 Score  Precision    Recall  Time Taken
Model                                                                                                   
XGBClassifier           0.866998           0.798996  0.925082  0.863628   0.862774  0.866998    0.727103
LGBMClassifier          0.866998           0.797351  0.925970  0.863353   0.862621  0.866998    3.537055
CatBoostClassifier      0.867330           0.796226  0.926156  0.863428   0.862851  0.867330    6.925837
RandomForestClassifier  0.850304           0.771002  0.903507  0.845437   0.844403  0.850304    3.483895
ExtraTreesClassifier    0.838917           0.761638  0.886299  0.834894   0.833153  0.838917    3.677341
NearestCentroid         0.736208           0.756904  0.823702  0.752840   0.807261  0.736208    0.060813
AdaBoostClassifier      0.846213           0.754677  0.897662  0.838907   0.839168  0.846213    1.162612
BaggingClassifier    

## 20 · FLAML AutoML Run

In [12]:
from flaml import AutoML

flaml_automl = AutoML()
flaml_automl.fit(
    X_train, y_train,
    task="classification", time_budget=60, metric="f1",
    estimator_list=['lgbm', 'rf', 'extra_tree', 'catboost'],
    seed=SEED, verbose=0
)
y_pred_flaml = flaml_automl.predict(X_test)
print(f"FLAML best: {flaml_automl.best_estimator}")
print(f"Accuracy : {accuracy_score(y_test, y_pred_flaml):.4f}")
print(f"F1       : {f1_score(y_test, y_pred_flaml, average='weighted'):.4f}")

FLAML best: catboost
Accuracy : 0.8664
F1       : 0.8623


## 21 · CatBoost, LightGBM, XGBoost

GPU auto-detected with CPU fallback.

In [13]:
import gc

def gpu_cleanup():
    gc.collect()
    try:
        import torch; torch.cuda.empty_cache()
    except Exception:
        pass

results = {}

# CatBoost
from catboost import CatBoostClassifier
try:
    cb = CatBoostClassifier(iterations=500, learning_rate=0.05, depth=6,
                            task_type="GPU", devices="0", verbose=0, random_seed=SEED)
    cb.fit(X_train, y_train)
except Exception:
    cb = CatBoostClassifier(iterations=500, learning_rate=0.05, depth=6,
                            verbose=0, random_seed=SEED)
    cb.fit(X_train, y_train)
results["CatBoost"] = cb.predict(X_test)
print(f"CatBoost  Acc: {accuracy_score(y_test, results['CatBoost']):.4f}  F1: {f1_score(y_test, results['CatBoost'], average='weighted'):.4f}")
gpu_cleanup()

# LightGBM
import lightgbm as lgbm
try:
    lg = lgbm.LGBMClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                              device="gpu", verbose=-1, random_state=SEED)
    lg.fit(X_train, y_train)
except Exception:
    lg = lgbm.LGBMClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                              verbose=-1, random_state=SEED)
    lg.fit(X_train, y_train)
results["LightGBM"] = lg.predict(X_test)
print(f"LightGBM  Acc: {accuracy_score(y_test, results['LightGBM']):.4f}  F1: {f1_score(y_test, results['LightGBM'], average='weighted'):.4f}")
gpu_cleanup()

# XGBoost
from xgboost import XGBClassifier
try:
    xgb_model = XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                               device="cuda", tree_method="hist", verbosity=0,
                               random_state=SEED, use_label_encoder=False, eval_metric="logloss")
    xgb_model.fit(X_train, y_train)
except Exception:
    xgb_model = XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                               tree_method="hist", verbosity=0,
                               random_state=SEED, use_label_encoder=False, eval_metric="logloss")
    xgb_model.fit(X_train, y_train)
results["XGBoost"] = xgb_model.predict(X_test)
print(f"XGBoost   Acc: {accuracy_score(y_test, results['XGBoost']):.4f}  F1: {f1_score(y_test, results['XGBoost'], average='weighted'):.4f}")
gpu_cleanup()

# Add baseline and FLAML
results["Logistic Regression"] = y_pred_bl
results["FLAML"] = y_pred_flaml

CatBoost  Acc: 0.8664  F1: 0.8622


LightGBM  Acc: 0.8678  F1: 0.8645


XGBoost   Acc: 0.8672  F1: 0.8636


## 22 · Top 3 Model Selection

In [14]:
model_scores = {}
for name, yp in results.items():
    model_scores[name] = {
        "Accuracy": round(accuracy_score(y_test, yp), 4),
        "F1": round(f1_score(y_test, yp, average='weighted'), 4),
        "Precision": round(precision_score(y_test, yp, average='weighted'), 4),
        "Recall": round(recall_score(y_test, yp, average='weighted'), 4),
    }

scores_df = pd.DataFrame(model_scores).T.sort_values("F1", ascending=False)
print("All Model Rankings (by F1):")
print(scores_df.to_string())
top3_names = scores_df.index[:3].tolist()
print(f"\nTop 3: {top3_names}")

All Model Rankings (by F1):
                     Accuracy      F1  Precision  Recall
LightGBM               0.8678  0.8645     0.8636  0.8678
XGBoost                0.8672  0.8636     0.8629  0.8672
FLAML                  0.8664  0.8623     0.8618  0.8664
CatBoost               0.8664  0.8622     0.8618  0.8664
Logistic Regression    0.8015  0.7813     0.7858  0.8015

Top 3: ['LightGBM', 'XGBoost', 'FLAML']


## 23 · Final Training and Evaluation of Top 3

In [15]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, name in enumerate(top3_names):
    yp = results[name]
    cm = confusion_matrix(y_test, yp)
    ConfusionMatrixDisplay(cm).plot(ax=axes[i], cmap='Blues')
    f1 = f1_score(y_test, yp, average='weighted')
    acc = accuracy_score(y_test, yp)
    axes[i].set_title(f"{name}\nAcc={acc:.4f} F1={f1:.4f}")

plt.suptitle("Top 3 — Confusion Matrices", fontsize=14)
plt.tight_layout()
plt.savefig('top3_confusion.png', dpi=100, bbox_inches='tight')
plt.show()

print("\nDetailed Classification Reports — Top 3:")
for name in top3_names:
    print(f"\n{'='*50}")
    print(f"  {name}")
    print('='*50)
    print(classification_report(y_test, results[name]))


Detailed Classification Reports — Top 3:

  LightGBM
              precision    recall  f1-score   support

           0       0.89      0.93      0.91      6803
           1       0.77      0.67      0.71      2242

    accuracy                           0.87      9045
   macro avg       0.83      0.80      0.81      9045
weighted avg       0.86      0.87      0.86      9045


  XGBoost
              precision    recall  f1-score   support

           0       0.89      0.94      0.91      6803
           1       0.77      0.66      0.71      2242

    accuracy                           0.87      9045
   macro avg       0.83      0.80      0.81      9045
weighted avg       0.86      0.87      0.86      9045


  FLAML
              precision    recall  f1-score   support

           0       0.89      0.94      0.91      6803
           1       0.78      0.65      0.71      2242

    accuracy                           0.87      9045
   macro avg       0.83      0.79      0.81      9045


## 24 · Error Analysis

In [16]:
best_name = top3_names[0]
best_pred = results[best_name]

errors = y_test != best_pred
error_rate = errors.mean()
print(f"Best model: {best_name}")
print(f"Error rate: {error_rate:.4f} ({errors.sum()} / {len(y_test)})")

# Errors by class
print(f"\nErrors by true class:")
error_df = pd.DataFrame({'true': y_test, 'pred': best_pred, 'error': errors})
print(error_df.groupby('true')['error'].agg(['sum', 'count', 'mean']).rename(
    columns={'sum': 'errors', 'count': 'total', 'mean': 'error_rate'}))

Best model: LightGBM
Error rate: 0.1322 (1196 / 9045)

Errors by true class:
      errors  total  error_rate
true                           
0        449   6803    0.066000
1        747   2242    0.333185


## 25 · Interpretation and Business Insight

- **Education and hours-per-week** are among the strongest predictors.
- **Capital-gain** is highly predictive but sparse (most values are 0).
- **Marital status** captures household income effects.
- **Age** shows a non-linear relationship — income peaks in 40s-50s.
- Features like race and sex are predictive but raise fairness concerns.

## 26 · Limitations

1. Census data from 1994 — income threshold ($50K) is outdated.
2. Class imbalance (~24% positive) affects minority-class recall.
3. Missing values dropped rather than imputed.
4. No temporal dimension — snapshot data.
5. Fairness issues with protected attributes (race, sex).

## 27 · How to Improve This Project

1. Impute missing values instead of dropping rows.
2. Apply SMOTE or class weights for imbalance.
3. Add fairness-aware evaluation (disparate impact, equalized odds).
4. Use SHAP for feature importance explanation.
5. Try TabPFN-v2 for comparison on this medium-sized dataset.

## 28 · Production Considerations

- Model monitoring for distribution drift (demographics change over time).
- Fairness auditing required before deployment.
- A/B testing against rule-based systems.
- Prediction explanations (SHAP/LIME) for regulatory compliance.

## 29 · Common Mistakes

1. Using accuracy alone on imbalanced data.
2. Not checking for data leakage (e.g., capital-gain may leak income).
3. Label encoding high-cardinality categoricals without thought.
4. Ignoring fairness — deploying without bias audit.
5. Not stratifying the train/test split.

## 30 · Mini Challenge / Exercises

1. Drop race and sex — does model performance change?
2. Apply SMOTE and compare recall on the minority class.
3. Use SHAP to explain individual predictions.
4. Build a fairness dashboard comparing predictions across groups.

## 31 · Final Summary / Key Takeaways

1. Adult Income is a classic benchmark for binary classification.
2. CatBoost/LightGBM/XGBoost typically achieve ~86-87% accuracy.
3. Class imbalance requires careful metric selection (F1, ROC-AUC).
4. Feature importance reveals education and work hours as key drivers.
5. Fairness considerations are critical for real deployment.

## Save Metrics

In [17]:
metrics_out = {}
for name, yp in results.items():
    metrics_out[name] = {
        "accuracy": round(float(accuracy_score(y_test, yp)), 4),
        "f1": round(float(f1_score(y_test, yp, average='weighted')), 4),
        "precision": round(float(precision_score(y_test, yp, average='weighted')), 4),
        "recall": round(float(recall_score(y_test, yp, average='weighted')), 4),
    }

with open('metrics.json', 'w') as f:
    json.dump(metrics_out, f, indent=2)
print("Metrics saved.")
print(json.dumps(metrics_out, indent=2))

Metrics saved.
{
  "CatBoost": {
    "accuracy": 0.8664,
    "f1": 0.8622,
    "precision": 0.8618,
    "recall": 0.8664
  },
  "LightGBM": {
    "accuracy": 0.8678,
    "f1": 0.8645,
    "precision": 0.8636,
    "recall": 0.8678
  },
  "XGBoost": {
    "accuracy": 0.8672,
    "f1": 0.8636,
    "precision": 0.8629,
    "recall": 0.8672
  },
  "Logistic Regression": {
    "accuracy": 0.8015,
    "f1": 0.7813,
    "precision": 0.7858,
    "recall": 0.8015
  },
  "FLAML": {
    "accuracy": 0.8664,
    "f1": 0.8623,
    "precision": 0.8618,
    "recall": 0.8664
  }
}
